# 🚀 R2AI2026 — Phương án A+B: Hybrid Search + BGE-Reranker

**Mục tiêu:** Cải thiện điểm từ 0.3976 lên 0.45-0.50

**Quy trình:**
1. ✅ Setup môi trường & mount Drive
2. ✅ Cài thư viện
3. ✅ Giải nén code
4. ✅ Phương án A — Index (BM25 + ChromaDB)
5. ✅ Phương án A — Hybrid Retrieve (cập nhật retrieval.db)
6. ✅ Phương án B — Rerank + Retune → submission.zip

⏱️ **Thời gian ước tính:** ~2.5 tiếng trên T4 GPU

> **LƯU Ý:** KHÔNG cần chạy lại LLM (Gemma-9B). Câu trả lời giữ nguyên từ `results_partial.jsonl`!

## Cell 1 — Kiểm tra GPU & Mount Drive

In [ ]:
# ── Cell 1: Kiểm tra GPU & Mount Drive ──────────────────────────────────────
import subprocess, os

# Kiểm tra GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('🖥️  GPU info:')
print(result.stdout.strip())

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục artifacts trên Drive
DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'\n📁 Drive dir: {DRIVE_DIR}')
print('✅ Cell 1 Done!')

## Cell 2 — Cài thư viện

In [ ]:
# ── Cell 2: Cài thư viện ────────────────────────────────────────────────────
print('📦 Đang cài thư viện...')
import subprocess, sys

packages = [
    'chromadb==0.5.23',
    'sentence-transformers>=2.7.0',
    'rank-bm25',
    'underthesea',
    'tqdm',
    'numpy',
]

for pkg in packages:
    print(f'  Installing {pkg}...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                   capture_output=True)

print('\n✅ Cell 2 Done! Tất cả thư viện đã cài xong.')

## Cell 3 — Upload & Giải nén Project

In [ ]:
# ── Cell 3: Upload & Giải nén Project ───────────────────────────────────────
# HƯỚNG DẪN: Upload các file sau lên Google Drive vào thư mục R2AI_Artifacts:
#   - sme_rag_clean.zip  (code project)
#   - results_partial.jsonl  (câu trả lời LLM cũ)
#   - R2AIStage1DATA.json    (2000 câu hỏi)

import os, subprocess

DRIVE_DIR      = '/content/drive/MyDrive/R2AI_Artifacts'
PROJECT_ZIP    = f'{DRIVE_DIR}/sme_rag_clean.zip'
ANSWERS_FILE   = f'{DRIVE_DIR}/results_partial.jsonl'
QUESTIONS_FILE = f'{DRIVE_DIR}/R2AIStage1DATA.json'
WORK_DIR       = '/content/sme-legal-assistant'

# Kiểm tra file tồn tại
for f in [PROJECT_ZIP, ANSWERS_FILE, QUESTIONS_FILE]:
    status = '✅' if os.path.exists(f) else '❌ THIẾU'
    print(f'{status} {f}')

# Giải nén project
if os.path.exists(WORK_DIR):
    subprocess.run(['rm', '-rf', WORK_DIR])
os.makedirs(WORK_DIR, exist_ok=True)
subprocess.run(['unzip', '-q', PROJECT_ZIP, '-d', WORK_DIR])
print(f'\n📂 Project giải nén tại: {WORK_DIR}')

# Copy file answers và câu hỏi vào project
import shutil
os.makedirs(f'{WORK_DIR}/artifacts/output', exist_ok=True)
os.makedirs(f'{WORK_DIR}/artifacts/cache', exist_ok=True)
os.makedirs(f'{WORK_DIR}/data', exist_ok=True)

shutil.copy2(ANSWERS_FILE,   f'{WORK_DIR}/artifacts/output/results_partial.jsonl')
shutil.copy2(QUESTIONS_FILE, f'{WORK_DIR}/data/R2AIStage1DATA.json')
print('✅ Đã copy answers + questions vào project')

# Thêm src vào Python path
import sys
sys.path.insert(0, f'{WORK_DIR}/src')
os.chdir(WORK_DIR)
print(f'✅ Working dir: {os.getcwd()}')
print('✅ Cell 3 Done!')

## Cell 4 — Phương án A (Phần 1): INGEST dữ liệu luật

In [ ]:
# ── Cell 4: Phương án A — Ingest (Tải + chunking dữ liệu luật) ───────────────
# ⏱️ Ước tính: 10-15 phút
# Bỏ qua nếu đã có file artifacts/raw/ từ lần chạy trước

import os
chunks_file = 'artifacts/raw/chunks.jsonl'

if os.path.exists(chunks_file):
    import json
    count = sum(1 for _ in open(chunks_file))
    print(f'✅ Chunks đã tồn tại: {count} chunks — Bỏ qua Ingest!')
else:
    print('🔄 Bắt đầu Ingest...')
    !python run.py ingest
    print('\n✅ Cell 4 Done!')

# Backup chunks lên Drive
DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts'
if os.path.exists(chunks_file):
    import shutil
    shutil.copy2(chunks_file, f'{DRIVE_DIR}/chunks.jsonl')
    print(f'☁️  Chunks saved to Drive')

## Cell 5 — Phương án A (Phần 2): INDEX — Tạo BM25 + ChromaDB

In [ ]:
# ── Cell 5: Phương án A — Index (BM25 + ChromaDB) ───────────────────────────
# ⏱️ Ước tính: 30-45 phút (tải embedding model + embed toàn bộ kho luật)

import os, shutil
DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts'

# Kiểm tra nếu đã có index sẵn từ Drive
drive_index = f'{DRIVE_DIR}/index_built.zip'

if os.path.exists(drive_index):
    print('📦 Tìm thấy index cũ trên Drive, đang giải nén...')
    import subprocess
    subprocess.run(['unzip', '-q', '-o', drive_index, '-d', 'artifacts/'])
    print('✅ Index đã restore từ Drive — Bỏ qua build!')
else:
    print('🔄 Bắt đầu xây dựng Index (BM25 + ChromaDB)...')
    print('   Embedding model: mainguyen9/vietlegal-e5')
    print('   Đây là bước tốn nhiều thời gian nhất (~45 phút)...')
    !python run.py index --device cuda --reset

    # Backup index lên Drive để lần sau không cần build lại
    print('\n☁️  Đang nén và backup index lên Drive...')
    subprocess.run(['zip', '-r', '-q', drive_index,
                    'artifacts/index/', 'artifacts/raw/'])
    print(f'✅ Index đã backup: {drive_index}')

print('\n✅ Cell 5 Done!')

## Cell 6 — Phương án A (Phần 3): RETRIEVE — Hybrid Search cho 2000 câu

In [ ]:
# ── Cell 6: Phương án A — Hybrid Retrieve ───────────────────────────────────
# ⏱️ Ước tính: 60-90 phút
# Tạo retrieval.db MỚI với điểm Hybrid (BM25 + Dense Embedding)
# Tự động resume nếu bị ngắt giữa chừng!

import os, shutil, subprocess
DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts'
DB_PATH   = 'artifacts/cache/retrieval.db'
DRIVE_DB  = f'{DRIVE_DIR}/retrieval_hybrid.db'

# Nếu có retrieval.db từ Drive thì restore (resume từ lần chạy dở)
if os.path.exists(DRIVE_DB) and not os.path.exists(DB_PATH):
    os.makedirs('artifacts/cache', exist_ok=True)
    shutil.copy2(DRIVE_DB, DB_PATH)
    print(f'📦 Restored retrieval.db từ Drive')

print('🔄 Bắt đầu Hybrid Retrieval cho 2000 câu...')
print('   BM25 + ChromaDB (vietlegal-e5) → RRF fusion → retrieval.db')
print('   Tự động lưu checkpoint sau mỗi batch...')

# Chạy retrieve (có --no-reranker vì reranker sẽ chạy riêng ở Cell 7)
result = subprocess.run(
    ['python', 'run.py', 'retrieve',
     '--questions', 'data/R2AIStage1DATA.json',
     '--device', 'cuda',
     '--no-reranker'],  # Không rerank ở đây, để Cell 7 làm
    capture_output=False
)

# Backup retrieval.db lên Drive
if os.path.exists(DB_PATH):
    shutil.copy2(DB_PATH, DRIVE_DB)
    print(f'\n☁️  retrieval_hybrid.db saved to Drive: {DRIVE_DB}')

# Kiểm tra kết quả
import sqlite3
conn = sqlite3.connect(DB_PATH)
count = conn.execute('SELECT COUNT(*) FROM retrieval_cache').fetchone()[0]
conn.close()
print(f'\n📊 retrieval.db: {count}/2000 câu đã được retrieve')
print('\n✅ Cell 6 Done!')

## Cell 7 — Phương án B: BGE-Reranker + Retune → submission.zip

In [ ]:
# ── Cell 7: Phương án B — BGE-Reranker + Retune ─────────────────────────────
# ⏱️ Ước tính: 20-30 phút
# Đọc retrieval.db mới, chạy BGE-Reranker, xuất submission_reranked.zip

import subprocess, os, shutil
DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts'

print('🤖 Bắt đầu BGE-Reranker + Retune...')
print('   Model: BAAI/bge-reranker-v2-m3')
print('   Config: HIGH_CONF=0.62, SAFE=0.52, MAX_ART=3')
print('   Checkpoint mỗi 100 câu → tự động save lên Drive')
print()

result = subprocess.run(
    ['python', 'rerank_retune.py',
     '--high-conf', '0.62',
     '--safe',      '0.52',
     '--min-art',   '0',
     '--max-art',   '3',
     '--device',    'cuda',
     '--batch-size','64'],
    capture_output=False
)

# Kiểm tra output
zip_path = 'artifacts/output/submission_reranked.zip'
if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f'\n🏆 submission_reranked.zip: {size_mb:.1f} MB')
    print('\n✅ Cell 7 Done! File đã được lưu lên Drive tự động.')
    print(f'   📥 Tải về từ Drive: {DRIVE_DIR}/submission_reranked.zip')
else:
    print('❌ Không tìm thấy file output!')

## Cell 8 — (Tùy chọn) Thử các mức threshold khác

In [ ]:
# ── Cell 8: Thử threshold khác (KHÔNG cần chạy lại Cell 5-6) ────────────────
# Sau khi đã có retrieval.db mới, chỉ cần đổi số và chạy lại cell này

import subprocess, os, shutil
DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts'

# ── Chỉnh các số này ──
HIGH_CONF = 0.63
SAFE      = 0.50
MAX_ART   = 3
# ─────────────────────

out_name = f'submission_{str(HIGH_CONF).replace(".","")}_3_{str(SAFE).replace(".","")}.zip'
print(f'🔧 Retune với HIGH_CONF={HIGH_CONF}, SAFE={SAFE}, MAX_ART={MAX_ART}')

result = subprocess.run(
    ['python', 'rerank_retune.py',
     '--high-conf', str(HIGH_CONF),
     '--safe',      str(SAFE),
     '--min-art',   '0',
     '--max-art',   str(MAX_ART),
     '--device',    'cuda',
     '--batch-size','64'],
    capture_output=False
)

src = 'artifacts/output/submission_reranked.zip'
if os.path.exists(src):
    dst = f'{DRIVE_DIR}/{out_name}'
    shutil.copy2(src, dst)
    print(f'\n✅ Saved: {dst}')

## 📋 Tóm tắt

| Cell | Tác vụ | Thời gian |
|------|---------|----------|
| 1 | GPU check + Mount Drive | 1 phút |
| 2 | Cài thư viện | 3-5 phút |
| 3 | Giải nén project | 2 phút |
| 4 | Ingest dữ liệu luật | 10-15 phút |
| 5 | **Index BM25 + ChromaDB** | **30-45 phút** |
| 6 | **Hybrid Retrieve 2000 câu** | **60-90 phút** |
| 7 | **BGE-Reranker + Retune** | **20-30 phút** |
| 8 | Thử threshold khác | 20-30 phút |

**Tổng: ~2-3 tiếng** (không cần chạy LLM 40 tiếng!)

### Files cần upload lên Drive:
1. `sme_rag_clean.zip` — Code project (tạo từ Cell hướng dẫn bên dưới)
2. `results_partial.jsonl` — Câu trả lời LLM đã có
3. `R2AIStage1DATA.json` — 2000 câu hỏi thi

## Cell 0 — (Chạy trên máy bạn trước) Tạo file zip clean để upload

In [ ]:
# ── Cell 0: TẠO FILE ZIP (chạy trên máy tính của bạn, KHÔNG phải Colab) ──────
# Lệnh này tạo file sme_rag_clean.zip sạch sẽ để upload lên Drive
#
# Chạy trong PowerShell trên máy bạn:
#
#   cd D:\Project\sme-legal-assistant
#   $exclude = @('artifacts', '__pycache__', '.git', '*.zip', '*.db', '*.jsonl', '*.json')
#   Get-ChildItem -Recurse | Where-Object {...} | ...
#
# Hoặc đơn giản hơn: Mình sẽ chạy lệnh này cho bạn bằng script Python:

print('Đây là cell hướng dẫn — chạy script tạo zip trên máy tính của bạn')
print('Xem cell bên dưới để chạy trên Colab nếu đã upload zip lên Drive rồi')